<a href="https://colab.research.google.com/github/zeinab-salah/COSC726-Agentic-Artificial-Intelligence/blob/main/Week_2_Lab_1_llm_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Two DELIBERATELY DIFFERENT teaching tokenizers. Neither is authoritative — that is the point.
def greedy_split(text, vocab):
    vocab = sorted(vocab, key=len, reverse=True)
    text, out, i = text.lower(), [], 0
    while i < len(text):
        for v in vocab:
            if v and text.startswith(v.lower(), i):
                out.append(v); i += len(v); break
        else:
            out.append(text[i]); i += 1
    return out

TOK_A = ["order","agent","the","credit","policy","late","1043","ic","ing","ed"," ","-","A","#"]
TOK_B = ["order","ag","ent","the","cred","it","pol","icy","late","10","43","ic","ing","ed"," ","-","A","#"]

for label, vocab in [("A", TOK_A), ("B", TOK_B)]:
    toks = greedy_split("order A-1043", vocab)
    print(f"tokenizer {label}: {len(toks)} tokens  {toks}")



tokenizer A: 5 tokens  ['order', ' ', 'A', '-', '1043']
tokenizer B: 6 tokens  ['order', ' ', 'A', '-', '10', '43']


In [2]:
samples = {
    "English":    "the late credit policy",
    "Arabic":     "الذكاء الاصطناعي القائم على الوكلاء",   # counted as raw characters by our toy tokenizer
    "Identifier": "customer_order_A-1043",
    "JSON":       '{"status":"pending_approval"}',
}
print(f"{'input':<12}{'tok A':>7}{'tok B':>7}   provenance you MUST record")
print("-" * 60)
for name, text in samples.items():
    a = len(greedy_split(text, TOK_A))
    b = len(greedy_split(text, TOK_B))
    print(f"{name:<12}{a:>7}{b:>7}   (tokenizer + version + the exact text)")
print("\nObservation 1: which input is most expensive, and why? A count WITHOUT")
print("its tokenizer/version is not reproducible evidence.")

print("**Observation 1.** The Arabic string is the most expensive input under both tokenizers. Neither vocab contains any Arabic word or subword pieces — every entry is an English fragment (`order`, `agent`, `credit`, ...) — so `greedy_split` never finds a match for an Arabic character and falls through to emitting it as a single-character token every time. That means the Arabic input costs roughly one token per character, while the English and JSON inputs reuse multi-character vocab entries and cost far fewer tokens per character of text. This isn't a property of Arabic as a language; it's a property of a vocab that was never built with Arabic subwords in it — a tokenizer trained on Arabic morphemes would compress it just as efficiently as English is compressed here.")

input         tok A  tok B   provenance you MUST record
------------------------------------------------------------
English           7      9   (tokenizer + version + the exact text)
Arabic           35     35   (tokenizer + version + the exact text)
Identifier       14     15   (tokenizer + version + the exact text)
JSON             27     27   (tokenizer + version + the exact text)

Observation 1: which input is most expensive, and why? A count WITHOUT
its tokenizer/version is not reproducible evidence.
**Observation 1.** The Arabic string is the most expensive input under both tokenizers. Neither vocab contains any Arabic word or subword pieces — every entry is an English fragment (`order`, `agent`, `credit`, ...) — so `greedy_split` never finds a match for an Arabic character and falls through to emitting it as a single-character token every time. That means the Arabic input costs roughly one token per character, while the English and JSON inputs reuse multi-character vocab entri

In [4]:
def count_tokens(text, vocab):
    return len(greedy_split(text, vocab))

def prepare_context(messages, context_limit, reserved_output, vocab, strategy="drop_oldest"):
    budget = context_limit - reserved_output
    total = lambda ms: sum(count_tokens(m, vocab) for m in ms)
    if strategy == "reject":
        if total(messages) > budget:
            return {"rejected": True, "reason": f"input {total(messages)} > budget {budget}",
                    "kept": [], "dropped": []}
        return {"rejected": False, "reason": "", "kept": list(messages), "dropped": []}
    kept, dropped = list(messages), []
    while total(kept) > budget and len(kept) > 1:
        dropped.append(kept.pop(1))            # keep messages[0] (system) always
    return {"rejected": total(kept) > budget, "reason": "", "kept": kept, "dropped": dropped}

convo = ["SYSTEM: you are Layla's support agent",
         "USER: where is order #A1032?",
         "ASSISTANT: checking now",
         "USER: it is very late, do I get a credit?"]

print("REJECT policy (tight budget):")
print(" ", prepare_context(convo, context_limit=12, reserved_output=6, vocab=TOK_A, strategy="reject"))
print("\nDROP-OLDEST policy (same messages, logged):")
plan = prepare_context(convo, context_limit=40, reserved_output=6, vocab=TOK_A, strategy="drop_oldest")
print("  kept:   ", plan["kept"])
print("  dropped:", plan["dropped"], " <- LOGGED, never silent")
print("\nObservation 2: for Layla, which message would be UNSAFE to drop silently?")

print("**Observation 2.** The unsafe message to drop silently is `USER: it is very late, do I get a credit?`. If drop-oldest removes that turn without logging it, the transcript downstream loses the exact question the whole approval workflow exists to answer — the agent could keep replying as if this were still a plain status inquiry, and any later response about credit eligibility would look fabricated relative to what's actually left in context, since the customer's compensation question is no longer there to support it.")


REJECT policy (tight budget):
  {'rejected': True, 'reason': 'input 111 > budget 6', 'kept': [], 'dropped': []}

DROP-OLDEST policy (same messages, logged):
  kept:    ["SYSTEM: you are Layla's support agent"]
  dropped: ['USER: where is order #A1032?', 'ASSISTANT: checking now', 'USER: it is very late, do I get a credit?']  <- LOGGED, never silent

Observation 2: for Layla, which message would be UNSAFE to drop silently?
**Observation 2.** The unsafe message to drop silently is `USER: it is very late, do I get a credit?`. If drop-oldest removes that turn without logging it, the transcript downstream loses the exact question the whole approval workflow exists to answer — the agent could keep replying as if this were still a plain status inquiry, and any later response about credit eligibility would look fabricated relative to what's actually left in context, since the customer's compensation question is no longer there to support it.


In [5]:
import math, random

def sample_next(distribution, temperature, rng):
    if temperature <= 0:
        top = max(distribution.values())
        return sorted(k for k, v in distribution.items() if v == top)[0]
    scaled = {k: math.exp(math.log(v) / temperature) for k, v in distribution.items() if v > 0}
    z = sum(scaled.values())
    r, acc = rng.random(), 0.0
    for k in sorted(scaled):
        acc += scaled[k] / z
        if r <= acc:
            return k
    return sorted(scaled)[-1]

dist = {"Paris": 0.82, "London": 0.11, "Lyon": 0.05, "Rome": 0.02}
print("temperature 0 (greedy) over 5 seeds:",
      {sample_next(dist, 0.0, random.Random(s)) for s in range(5)}, " <- always the mode")
print("temperature 1.0 over 8 seeds:      ",
      [sample_next(dist, 1.0, random.Random(s)) for s in range(8)])
print("\nObservation 3: for Layla's ORDER-ID extraction, is high or low temperature safer — and how")
print("would you JUSTIFY the choice with evidence rather than taste?")

print("**Observation 3.** Low temperature (effectively 0 / greedy) is safer for order-ID extraction. This task has exactly one correct answer per input, so there's no legitimate diversity to explore — any temperature above 0 only adds a chance of returning a wrong or malformed ID. This is falsifiable, not a matter of taste: the same test used above shows that at temperature 0, `sample_next` returns the identical, correct token across every seed (`picks == {'Paris'}`), while at temperature 1.0 the picks vary across seeds. Applied to ID extraction you would run the extractor across many seeds at each temperature and measure the disagreement rate between runs; temperature 0 gives 0% disagreement by construction, which is the evidence for the choice.")

temperature 0 (greedy) over 5 seeds: {'Paris'}  <- always the mode
temperature 1.0 over 8 seeds:       ['Paris', 'Lyon', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris']

Observation 3: for Layla's ORDER-ID extraction, is high or low temperature safer — and how
would you JUSTIFY the choice with evidence rather than taste?
**Observation 3.** Low temperature (effectively 0 / greedy) is safer for order-ID extraction. This task has exactly one correct answer per input, so there's no legitimate diversity to explore — any temperature above 0 only adds a chance of returning a wrong or malformed ID. This is falsifiable, not a matter of taste: the same test used above shows that at temperature 0, `sample_next` returns the identical, correct token across every seed (`picks == {'Paris'}`), while at temperature 1.0 the picks vary across seeds. Applied to ID extraction you would run the extractor across many seeds at each temperature and measure the disagreement rate between runs; temperature 

In [6]:
def bare_call(context_messages):
    """Stateless: sees ONLY what it is handed, nothing else."""
    for m in context_messages:
        if "layla" in m.lower():
            return "I can see this is Layla."
    return "I do not know who you are."   # not broken — it was simply handed no history

class ConversationManager:
    """The application component that actually stores and reconstructs state."""
    def __init__(self): self.history = []
    def add(self, msg): self.history.append(msg)
    def call(self): return bare_call(self.history)

print("Bare call, no history handed in:  ", bare_call(["what is my name?"]))
mgr = ConversationManager()
mgr.add("Hi, I'm Layla, about order #A1032")
mgr.add("what is my name?")
print("Managed call, history reconstructed:", mgr.call())
print("\nObservation 4: name the component responsible for retention, deletion, and privacy here.")

print("**Observation 4.** `ConversationManager` is the component responsible — not the model call itself. `bare_call()` only ever sees the exact list of messages it's handed; it has no persistent memory of its own. It is `ConversationManager.history` that stores Layla's turns across calls, so it is also the component that must implement retention limits, deletion / right-to-be-forgotten requests, and access control over who can read that history — these are application-layer responsibilities, not model behavior.")

Bare call, no history handed in:   I do not know who you are.
Managed call, history reconstructed: I can see this is Layla.

Observation 4: name the component responsible for retention, deletion, and privacy here.
**Observation 4.** `ConversationManager` is the component responsible — not the model call itself. `bare_call()` only ever sees the exact list of messages it's handed; it has no persistent memory of its own. It is `ConversationManager.history` that stores Layla's turns across calls, so it is also the component that must implement retention limits, deletion / right-to-be-forgotten requests, and access control over who can read that history — these are application-layer responsibilities, not model behavior.


In [8]:
POLICY_DB = {"late_delivery": {"text": "Orders >3 days late qualify for a 10% credit.",
                               "evidence_id": "policy-late-delivery-v3"}}

def answer_with_grounding(question, require_evidence=True):
    hit = POLICY_DB.get("late_delivery") if "credit" in question or "late" in question else None
    if hit:
        return {"answer": hit["text"], "evidence_id": hit["evidence_id"], "abstained": False}
    if require_evidence:
        return {"answer": "Insufficient evidence — escalating to a human.",
                "evidence_id": None, "abstained": True}
    return {"answer": "Sure, Layla probably gets some money back!", "evidence_id": None, "abstained": False}

print("grounded  :", answer_with_grounding("is Layla's order eligible for a late credit?"))
print("no evidence, abstain :", answer_with_grounding("what is the CEO's home address?", require_evidence=True))
print("no evidence, ungrounded (UNSAFE):", answer_with_grounding("what is the CEO's home address?", require_evidence=False))
print("\nObservation 5: which of these three could safely drive an ACTION on Layla's account?")

print("**Observation 5.** Only the first case — the grounded answer, with `evidence_id: 'policy-late-delivery-v3'` and `abstained: False` — could safely drive an action. It's the only one where the claim is traceable to a specific, checkable source. The abstain case is safe in that it does nothing, but it explicitly can't drive an action since it states there is no evidence. The third case is the dangerous one: it produces a confident-sounding claim ('Layla probably gets some money back!') with no `evidence_id` at all — acting on it, e.g. issuing a credit, would mean taking a real, hard-to-reverse step based on an unverified guess.")

grounded  : {'answer': 'Orders >3 days late qualify for a 10% credit.', 'evidence_id': 'policy-late-delivery-v3', 'abstained': False}
no evidence, abstain : {'answer': 'Insufficient evidence — escalating to a human.', 'evidence_id': None, 'abstained': True}
no evidence, ungrounded (UNSAFE): {'answer': 'Sure, Layla probably gets some money back!', 'evidence_id': None, 'abstained': False}

Observation 5: which of these three could safely drive an ACTION on Layla's account?
**Observation 5.** Only the first case — the grounded answer, with `evidence_id: 'policy-late-delivery-v3'` and `abstained: False` — could safely drive an action. It's the only one where the claim is traceable to a specific, checkable source. The abstain case is safe in that it does nothing, but it explicitly can't drive an action since it states there is no evidence. The third case is the dangerous one: it produces a confident-sounding claim ('Layla probably gets some money back!') with no `evidence_id` at all — actin

In [9]:
def mock_generate(question):
    """A transparent, deterministic stand-in that returns a NORMALIZED response object."""
    return {
        "provider_model": "course-mock / mock-llm-v2",
        "request_id": "mock-7f31c2",
        "text": "Order #A1032 is 3 days late; a 10% credit applies (see policy-late-delivery-v3).",
        "usage": {"input_tokens": 118, "output_tokens": 36, "cached": 0, "reasoning": 12, "tool": 8},
        "finish_reason": "completed",
        "context_policy": "drop_oldest",
        "dropped_items": ["message-2"],
        "evidence_ids": ["policy-late-delivery-v3"],
        "abstained": False,
        "latency_ms": 640, "retries": 0,
    }

resp = mock_generate("is Layla's order eligible?")
for k, v in resp.items():
    print(f"  {k:<16} {v}")
print("\nObservation 6: which THREE fields would you need to (a) estimate cost, (b) reproduce")
print("the call, and (c) audit the evidence behind the answer?")

print("**Observation 6.**- **(a) Cost:** `usage` — it holds `input_tokens`, `output_tokens`, `cached`, `reasoning`, and `tool` counts, which is everything needed to compute a dollar cost against a pricing table.- **(b) Reproduce:** `provider_model` together with `context_policy` (and `dropped_items`) — knowing exactly which model/version served the request, and exactly which context-shaping policy was applied and what it dropped, is what lets you rebuild the same input conditions later.- **(c) Audit evidence:** `evidence_ids` together with `abstained` — `evidence_ids` shows which source claims are traceable to, and `abstained` shows whether the model itself flagged the answer as unsupported, which is essential for an audit trail.")

  provider_model   course-mock / mock-llm-v2
  request_id       mock-7f31c2
  text             Order #A1032 is 3 days late; a 10% credit applies (see policy-late-delivery-v3).
  usage            {'input_tokens': 118, 'output_tokens': 36, 'cached': 0, 'reasoning': 12, 'tool': 8}
  finish_reason    completed
  context_policy   drop_oldest
  dropped_items    ['message-2']
  evidence_ids     ['policy-late-delivery-v3']
  abstained        False
  latency_ms       640
  retries          0

Observation 6: which THREE fields would you need to (a) estimate cost, (b) reproduce
the call, and (c) audit the evidence behind the answer?
**Observation 6.**- **(a) Cost:** `usage` — it holds `input_tokens`, `output_tokens`, `cached`, `reasoning`, and `tool` counts, which is everything needed to compute a dollar cost against a pricing table.- **(b) Reproduce:** `provider_model` together with `context_policy` (and `dropped_items`) — knowing exactly which model/version served the request, and exactly which